In [1]:
import time
import sys
import random
import string

sys.path.append("/home/vardan/Desktop/src_model/cpp")

import my_module
from transliterate import unicodelowersplit as uls
from py_ex import JaroWinklerNGramSearch as JWS

In [2]:
stop_words = (
    "C c. city town "
    "ք ք․ քաղաք qaxaq "
    "город гор гор. "
    "village vlg vil v v. "
    "գ գ․ գյուղ "
    "համայնք "
    "с с. село "
    "п п. посёлок "
    "д д. деревня "
    "region provI@&*))(*nce "
    "մ մ․ մարզ "
    "ավան "
    "обл область край"
)

ln_code, stop_words = uls(stop_words)

print(ln_code)
print(stop_words)

en
['c', 'c.', 'city', 'town', 'q', 'q.', 'qaghaq', 'qaxaq', 'gorod', 'gor', 'gor.', 'village', 'vlg', 'vil', 'v', 'v.', 'g', 'g.', 'gyvough', 'hamaynq', 's', 's.', 'selo', 'p', 'p.', 'poslok', 'd', 'd.', 'derevnya', 'region', 'province', 'm', 'm.', 'marz', 'avan', 'obl', 'oblast', 'krai']


In [3]:
# ============================================================
# 2. C++ model
# ============================================================

mod_cpp = my_module.JaroNGramSearch(min_N=1,max_N=3,N_signif = 5)

s = time.perf_counter()
mod_cpp.fit(stop_words)
e = time.perf_counter()

cpp_fit_time = e - s


# ============================================================
# 3. Python model
# ============================================================

mod_py = JWS((1, 5))

s = time.perf_counter()
mod_py.fit(stop_words)
e = time.perf_counter()

py_fit_time = e - s


print("=" * 70)
print("FIT PERFORMANCE")
print("=" * 70)
print(f"C++ fit : {cpp_fit_time * 1000:.3f} ms")
print(f"Python  : {py_fit_time * 1000:.3f} ms")

FIT PERFORMANCE
C++ fit : 13.459 ms
Python  : 12.535 ms


In [4]:
import pprint
pprint.pprint(mod_cpp.FinalWordsData)

[('c', 1, [(3, 1.0, [0])]),
 ('c.',
  2,
  [(0, 0.07643853567642897, [1]),
   (3, 0.1018390003906107, [0]),
   (36, 0.8217224639329603, [0])]),
 ('city',
  4,
  [(3, 0.013766425267386199, [0]),
   (8, 0.012049629158314696, [1]),
   (18, 0.01498451149172067, [2]),
   (23, 0.013766425267386199, [3]),
   (38, 0.11107906447310832, [0]),
   (55, 0.11107906447310832, [1]),
   (90, 0.11107906447310832, [2]),
   (112, 0.30609790769793366, [0]),
   (127, 0.30609790769793366, [1])]),
 ('town',
  4,
  [(12, 0.012030542647155208, [3]),
   (13, 0.009468143939191915, [1]),
   (18, 0.014960776151649324, [0]),
   (21, 0.01960508637462923, [2]),
   (74, 0.1109031161699866, [1]),
   (89, 0.1109031161699866, [0]),
   (98, 0.1109031161699866, [2]),
   (143, 0.3056130511887073, [1]),
   (155, 0.3056130511887073, [0])]),
 ('q', 1, [(15, 1.0, [0])]),
 ('q.',
  2,
  [(0, 0.07697655888808332, [1]),
   (15, 0.09551716989874875, [0]),
   (78, 0.827506271213168, [0])]),
 ('qaghaq',
  6,
  [(1, 0.01139065512742563

In [5]:
pprint.pprint(mod_cpp.search('city'))

[('c', 1.0),
 ('city', 1.0),
 ('c.', 0.1018390003906107),
 ('vil', 0.02545504083139987),
 ('village', 0.005537247425545949)]


In [6]:
# ============================================================
# 4. Random test-data generator
# ============================================================

stop_words = (
    " city town "
    " քաղաք qaxaq "
    "город гор гор. "
    "village vlg vil"
    " գյուղ "
    "համայնք "
    " село "
    " посёлок "
    " деревня "
    "region province "
    " մարզ "
    "ավան "
    "обл область край"
)

ln_code, stop_words = uls(stop_words)


ALPHABET = string.ascii_lowercase + "./"


def mutate_word(word, changes):

    chars = list(word)

    for _ in range(changes):

        operation = random.choice(["replace", "insert"])

        if not chars:
            chars.append(random.choice(ALPHABET))
            continue

        if operation == "replace":

            pos = random.randrange(len(chars))

            chars[pos] = random.choice(ALPHABET)

        else:

            pos = random.randrange(len(chars) + 1)

            chars.insert(pos, random.choice(ALPHABET))

    return "".join(chars)


def generate_test_data(words, variants_min=10, variants_max=15):

    result = {}

    for word in words:

        count = random.randint(variants_min, variants_max)

        variants = set()

        while len(variants) < count:

            if len(word) == 1:
                changes = 1
            else:
                changes = random.randint(1, 3)

            variant = mutate_word(word, changes)

            if variant != word:
                variants.add(variant)

        result[word] = list(variants)

    return result


# ============================================================
# 5. Generate test data
# ============================================================

test_data = generate_test_data(stop_words, variants_min=10, variants_max=15)

In [7]:
print(test_data)

{'city': ['srty', 'cmty', 'cqty', 'ncuty', 'fito', 'cvty', 'cfty', 'citb', 'citys', 'citj', 'lcigty', 'biay', 'egtu'], 'town': ['toiq', 'tofwl', 'jtowpdn', 'towln', 'toxwjf', 'ectown', 'towni', 'tlown', 'ytown', 'tcdwn', 'tuwn', 'ktown', 'iiwn', 'knwn'], 'qaghaq': ['oqrghaqa', 'qaihaq', 'sqagharq', 'qanhaq', 'qdyghaq', 'vsghaq', 'qyaghaq', 'qatjaq', 'qagskhaq', 'qkghaq', 'qahgeaqb', 'qabuaq', 'qarhaq', 'qfghaqm', 'qkghmq'], 'qaxaq': ['qa.aq', 'qragzaq', 'qaxax', 'qtxkk', 'bqaxaq', 'qaquq', 'qazxqq', 'qaxaqo', 'eqbxaq', 'qaxuamq', 'qbaxbaq', 'qkuxaa'], 'gorod': ['gworodf', 'gxrod', 'gvored', 'torod', 'gjorok', 'goron', 'tgorowd', 'ponrhod', 'gororbe', 'gorx/d', 'gkorode.', 'goroxd', 'gerohid', 'go.ud', 'gorxood'], 'gor': ['gjou', 'go.e', 'gbor', 'go.', 'gur', 'gkogr', 'jfr', 'gvor', 'gqor', 'gdr', 'goyr', 'jgtr', 'gop', 'kor'], 'gor.': ['gor.w', 'gmrx', 'iokrs', 'gcor.', '.or.', 'gxorw', 'gor.l', 'ygor.', 'gorm', 'gyor.', 'cvoc.', 'lgor.'], 'village': ['kvifluage', 'vitwlage', 'vilxlagt

In [8]:
# ============================================================
# 6. Prepare queries
# ============================================================

test_queries = []

for original, variants in test_data.items():

    for variant in variants:

        test_queries.append(
            (original, variant)
        )


print()
print("=" * 70)
print("TEST DATA")
print("=" * 70)

print(
    f"Original words : {len(test_data)}"
)

print(
    f"Test queries   : {len(test_queries)}"
)


TEST DATA
Original words : 22
Test queries   : 278


In [9]:
# ============================================================
# 7. Search test
# ============================================================

times = []

found_count = 0


print()
print("=" * 70)
print("C++ SEARCH TEST")
print("=" * 70)


for i, (original, q) in enumerate(
    test_queries,
    1
):

    s = time.perf_counter()

    result = mod_cpp.search(q)

    e = time.perf_counter()

    elapsed = e - s

    times.append(elapsed)


    # --------------------------------------------------------
    # Ստուգում ենք՝ original բառը Top-5-ում կա՞
    # --------------------------------------------------------

    found = any(
        word == original
        for word, score in result
    )

    if found:
        found_count += 1


    # --------------------------------------------------------
    # Print
    # --------------------------------------------------------

    print(
        f"[{i:04}] "
        f"query={q!r:<15} "
        f"original={original!r:<15} "
        f"time={elapsed * 1000:>8.3f} ms "
        f"found={'YES' if found else 'NO'}"
    )

    for rank, (word, score) in enumerate(
        result,
        1
    ):

        print(
            f"       {rank}. "
            f"{word:<25} "
            f"score={score:.6f}"
        )


C++ SEARCH TEST
[0001] query='srty'          original='city'          time=   0.062 ms found=YES
       1. s                         score=1.000000
       2. city                      score=0.139830
       3. s.                        score=0.095517
       4. selo                      score=0.013146
       5. krai                      score=0.010882
[0002] query='cmty'          original='city'          time=   0.029 ms found=YES
       1. c                         score=1.000000
       2. m                         score=0.250000
       3. city                      score=0.153596
       4. c.                        score=0.101839
       5. m.                        score=0.025460
[0003] query='cqty'          original='city'          time=   0.021 ms found=YES
       1. c                         score=1.000000
       2. q                         score=0.250000
       3. city                      score=0.153596
       4. c.                        score=0.101839
       5. q.              

In [10]:
# ============================================================
# 8. Statistics
# ============================================================

total_queries = len(test_queries)

total_time = sum(times)

average_time = (
    total_time / total_queries
    if total_queries
    else 0
)

min_time = (
    min(times)
    if times
    else 0
)

max_time = (
    max(times)
    if times
    else 0
)

accuracy = (
    found_count / total_queries * 100
    if total_queries
    else 0
)


# ============================================================
# 9. Final result
# ============================================================

print()
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(
    f"Total queries : {total_queries}"
)

print(
    f"Found Top-5   : {found_count}"
)

print(
    f"Accuracy      : {accuracy:.2f}%"
)

print(
    f"Total time    : {total_time * 1000:.3f} ms"
)

print(
    f"Average time  : {average_time * 1000:.3f} ms"
)

print(
    f"Min time      : {min_time * 1000:.3f} ms"
)

print(
    f"Max time      : {max_time * 1000:.3f} ms"
)

print("=" * 70)


FINAL RESULTS
Total queries : 278
Found Top-5   : 270
Accuracy      : 97.12%
Total time    : 15.404 ms
Average time  : 0.055 ms
Min time      : 0.016 ms
Max time      : 2.240 ms
